# Total Site: Pulp Mill Utility-System Comparison
This notebook asks one decision question: what changes when the same pulp mill is viewed as a direct-integration problem versus a total-process and total-site utility-system problem, and how can the rendered graphs and raw graph payloads both be inspected through the packaged API?


## Case context
The packaged `pulp_mill.json` case represents a multi-area pulp mill with steam levels, hot-water utility interaction, and site-wide utility tradeoffs. `PinchWorkspace` manages named study cases, while each case stays a real `PinchProblem` with the standard `target`, `plot`, `summary_frame`, and `compare_to` interface. The same surface also supports selected-state direct or total-site reruns through `state_id` when the payload is stateful.


In [ ]:
import pandas as pd
import plotly.graph_objects as go

from OpenPinch import PinchWorkspace
from OpenPinch.utils import get_value

In [ ]:
workspace = PinchWorkspace(
    source="p_Varbanov et al.json",
    # source="pulp_mill.json",
    project_name="Site",
)
baseline = workspace.use_case("baseline")
summary = baseline.summary_frame()
catalog = baseline.plot.catalog()

{
    "cases": workspace.list_cases(),
    "active_case": workspace.active_case_name,
    "graph_count": len(catalog),
    "zone_tree_present": "zone_tree" in workspace.get_case_payload("baseline"),
}

In [ ]:
direct_target_name = "Site/Direct Integration"
total_process_target_name = "Site/Total Process Target"
total_site_target_name = "Site/Total Site Target"

target_order = [
    direct_target_name,
    total_process_target_name,
    total_site_target_name,
]
target_labels = {
    direct_target_name: "Direct Integration",
    total_process_target_name: "Total Process Target",
    total_site_target_name: "Total Site Target",
}

## Base solve and graph inventory
Start by confirming the scope ladder and the graph families already available through the built-in `plot.catalog()` output. The direct-integration target gives the local process view, while the total-site target introduces the site-wide utility graphs.


In [ ]:
catalog.loc[
    catalog["Target"].isin([direct_target_name, total_site_target_name]),
    ["Target", "Graph Type", "Graph Name"],
].drop_duplicates().reset_index(drop=True)

## Selected-state total-site call surface
This packaged case is scalar, so the targets do not move by state, but the selected-state API is the supported way to rerun one indirect integration state from a multistate payload.


In [ ]:
selected_state_total_site = baseline.target.indirect_heat_integration(state_id="0")
selected_state_summary = baseline.summary_frame()
selected_state_summary.loc[
    selected_state_summary["Target"].isin([direct_target_name, total_site_target_name]),
    ["Target", "State ID", "Hot Utility Target", "Cold Utility Target", "Heat Recovery"],
]


## Compare direct, total-process, and total-site scopes
Read these three rows first. They answer whether the same plant looks materially different once utility-system interaction is reconciled at the total-process and total-site levels.


In [ ]:
target_rows = summary.loc[
    summary["Target"].isin(target_order),
    [
        "Target",
        "Hot Utility Target",
        "Cold Utility Target",
        "Heat Recovery",
        "Hot Pinch",
        "Cold Pinch",
    ],
].copy()
target_rows["Scope"] = target_rows["Target"].map(target_labels)
target_rows = target_rows.set_index("Target").loc[target_order].reset_index(drop=True)
target_rows

In [ ]:
scope_rows = {row["Scope"]: row for _, row in target_rows.iterrows()}
comparison_pairs = [
    ("Total Process - Direct", "Total Process Target", "Direct Integration"),
    ("Total Site - Direct", "Total Site Target", "Direct Integration"),
]

scope_deltas = []
for label, lhs, rhs in comparison_pairs:
    left = scope_rows[lhs]
    right = scope_rows[rhs]
    scope_deltas.append(
        {
            "Comparison": label,
            "Hot Utility Delta": left["Hot Utility Target"]
            - right["Hot Utility Target"],
            "Cold Utility Delta": left["Cold Utility Target"]
            - right["Cold Utility Target"],
            "Heat Recovery Delta": left["Heat Recovery"] - right["Heat Recovery"],
            "Hot Pinch Delta": left["Hot Pinch"] - right["Hot Pinch"],
            "Cold Pinch Delta": left["Cold Pinch"] - right["Cold Pinch"],
        }
    )

pd.DataFrame(scope_deltas)

In [ ]:
target_metrics = target_rows.melt(
    id_vars="Scope",
    value_vars=["Hot Utility Target", "Cold Utility Target", "Heat Recovery"],
    var_name="Metric",
    value_name="Value",
)

metric_colours = {
    "Hot Utility Target": "#c0392b",
    "Cold Utility Target": "#2471a3",
    "Heat Recovery": "#138d75",
}

scope_comparison_fig = go.Figure()
for metric in ["Hot Utility Target", "Cold Utility Target", "Heat Recovery"]:
    metric_rows = target_metrics.loc[target_metrics["Metric"] == metric]
    scope_comparison_fig.add_bar(
        name=metric,
        x=metric_rows["Scope"],
        y=metric_rows["Value"],
        marker_color=metric_colours[metric],
        text=[f"{value:,.0f}" for value in metric_rows["Value"]],
        textposition="outside",
    )

scope_comparison_fig.update_layout(
    barmode="group",
    title="Utility targets and heat recovery across scope levels",
    xaxis_title="Scope level",
    yaxis_title="kW",
)
scope_comparison_fig

## Which zones drive the site picture?
Before reading total-site graphs, look at the direct-integration zone rows. This shows which areas dominate hot utility demand, cold utility demand, and recovered heat before those effects are aggregated into a site-level utility system answer.


In [ ]:
zone_rows = summary.loc[
    summary["Target"].str.endswith("/Direct Integration")
    & (summary["Target"] != direct_target_name),
    ["Target", "Hot Utility Target", "Cold Utility Target", "Heat Recovery"],
].copy()
zone_rows["Zone"] = zone_rows["Target"].str.replace(
    "/Direct Integration", "", regex=False
)
zone_rows["Net Utility Demand"] = (
    zone_rows["Hot Utility Target"] + zone_rows["Cold Utility Target"]
)

leader_tables = []
for metric in ["Hot Utility Target", "Cold Utility Target", "Heat Recovery"]:
    leaders = zone_rows.nlargest(5, metric)[["Zone", metric]].reset_index(drop=True)
    leaders.index = leaders.index + 1
    leaders = leaders.rename_axis("Rank").reset_index()
    leaders["Metric"] = metric
    leader_tables.append(leaders[["Metric", "Rank", "Zone", metric]])

pd.concat(leader_tables, ignore_index=True)

## Render the key graphs in scope order
Read the local-process GCC first, then move to the total-site profiles and the Site Utility Grand Composite Curve. Because the case is a real `PinchProblem`, the standard `plot` accessor is enough for all three figures.


In [ ]:
# baseline.plot.grand_composite_curve(zone_name=direct_target_name)
baseline.plot.grand_composite_curve(zone_name="Process A - chemical plant")

In [ ]:
baseline.plot.total_site_profiles(zone_name=total_site_target_name)

In [ ]:
baseline.plot.site_utility_grand_composite_curve(zone_name=total_site_target_name)

## Inspect the raw graph payloads
Rendered figures are one view. The packaged `plot.get_graph_data()` method exposes the raw graph payloads directly when a GUI or external layer needs serialized graph records.


In [ ]:
graph_payload = baseline.plot.get_graph_data()

graph_payload_overview = (
    catalog.loc[
        catalog["Target"].isin([direct_target_name, total_site_target_name]),
        ["Target", "Graph Type", "Graph Name", "Zone Address"],
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)
graph_payload_overview

In [ ]:
selected_payloads = {
    "Direct GCC": baseline.plot.grand_composite_curve(
        zone_name=direct_target_name,
        return_graph_data=True,
    ),
    "Total Site Profiles": baseline.plot.total_site_profiles(
        zone_name=total_site_target_name,
        return_graph_data=True,
    ),
    "Site Utility Grand Composite Curve": baseline.plot.site_utility_grand_composite_curve(
        zone_name=total_site_target_name,
        return_graph_data=True,
    ),
}

pd.DataFrame(
    [
        {
            "Graph": label,
            "Graph Type": payload["type"],
            "Segment Count": len(payload.get("segments", [])),
            "First Segment Title": payload["segments"][0].get("title")
            if payload.get("segments")
            else None,
            "First Segment Keys": ", ".join(sorted(payload["segments"][0].keys()))
            if payload.get("segments")
            else None,
        }
        for label, payload in selected_payloads.items()
    ]
)

## Local versus site cogeneration screens
Use named copies of the baseline case when you want side-by-side low-level targeting objects without losing the original study case.


In [ ]:
bleach_problem = workspace.copy_case(
    "baseline", "bleaching_cogeneration_screen", activate=False
)
bleach_problem.target()
cogen_bleach = bleach_problem.target.cogeneration(zone_name="Bleaching")

site_problem = workspace.copy_case(
    "baseline", "site_cogeneration_screen", activate=False
)
site_problem.target()
cogeneration_target = site_problem.target.cogeneration(
    options={
        "DO_TURBINE_WORK": True,
        "base_target_type": "Total Site Target",
    }
)

pd.DataFrame(
    [
        {
            "Screen": "Bleaching local screen",
            "Target": cogen_bleach.name,
            "Hot Utility Target": get_value(cogen_bleach.hot_utility_target),
            "Cold Utility Target": get_value(cogen_bleach.cold_utility_target),
            "Heat Recovery": get_value(cogen_bleach.heat_recovery_target),
            "Power Cogeneration Target": get_value(cogen_bleach.work_target),
            "Turbine Efficiency Target": get_value(
                cogen_bleach.turbine_efficiency_target
            ),
        },
        {
            "Screen": "Total-site screen",
            "Target": cogeneration_target.name,
            "Hot Utility Target": get_value(cogeneration_target.hot_utility_target),
            "Cold Utility Target": get_value(cogeneration_target.cold_utility_target),
            "Heat Recovery": get_value(cogeneration_target.heat_recovery_target),
            "Power Cogeneration Target": get_value(cogeneration_target.work_target),
            "Turbine Efficiency Target": get_value(
                cogeneration_target.turbine_efficiency_target
            ),
        },
    ]
)

## Interpretation
In this pulp mill, the direct-integration row gives the local process answer, but the total-process and total-site rows reveal how much that answer changes once utility-system interaction is reconciled. The direct GCC explains process bottlenecks; the total-site profiles show the broader hot and cold utility balance; the SUGCC is the graph to read when judging steam-system leverage and whether a site-wide utility change is worth deeper study. `PinchWorkspace` then keeps those named study cases together while the raw graph payloads remain available through the standard plot API.
